# Import packages :

In [ ]:
import utils
from utils import *

from sklearn.model_selection import train_test_split

In [6]:
reset_val_dir()

# Dataset and Data Augmentation :

In [7]:


train_transform = YoloCompatibleTransform(resized_img_width=utils.RESIZED_IMG_WIDTH, 
                                          resized_img_height=utils.RESIZED_IMG_HEIGHT, train=True)
test_transform = YoloCompatibleTransform(resized_img_width=utils.RESIZED_IMG_WIDTH, 
                                         resized_img_height=utils.RESIZED_IMG_HEIGHT, train=False)


# Read full CSV that has 2 columns: image name (LXXXXXXX.JPG) and label name (LXXXXXXX.txt)
full_df = pd.read_csv("./link_images_labels_YOLO_v3.csv")
# Split into train/validation
train_df, val_df = train_test_split(full_df, test_size=0.1)

# Create datasets
train_dataset = Dataset(
	df=train_df,
	image_dir="../dataset_project_iapr2025/train/",
	label_dir="./labels/",
	anchors=utils.ANCHORS,
	transform=train_transform
)
val_dataset = Dataset(
	df=val_df,
	image_dir="../dataset_project_iapr2025/train/",  # Same as train
	label_dir="./labels/",
	anchors=utils.ANCHORS,
	transform=test_transform
)

# Create dataloaders
train_loader = torch.utils.data.DataLoader(
	train_dataset,
	batch_size=BATCH_SIZE,
	shuffle=True,
	pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
	val_dataset,
	batch_size=BATCH_SIZE,
	shuffle=False,
	pin_memory=True,
)

In [11]:
num_images = len(train_loader.dataset)
print(f'Total number of images: {num_images}')

Total number of images: 81


# Model :

# Loss :

# Csv file generation

In [ ]:
# ============ TEST SETUP FOR SUBMISSION ============ #


test_dir = "../dataset_project_iapr2025/test"
test_files = sorted([f for f in os.listdir(test_dir) if f.endswith(".JPG")])

test_dataset = YOLOTestDataset(
	img_dir=test_dir,
	filenames=test_files,
	transform=test_transform
)



# Functions training and evaluation :

In [ ]:
def training_loop(loader, model, optimizer, loss_fn, scaled_anchors=SCALED_ANCHORS, device=DEVICE):
	"""
	Function to train the model on one epoch (of the training loader)
	"""
	
	# Creating a progress bar 
	progress_bar = tqdm(loader, leave=True)
	# Initializing a list to store the losses 
	losses = []
	model.train()
	# Iterating over the training data 
	for _, (x, y) in enumerate(progress_bar): 
		x = x.to(device) # x is a tensor of shape (batch_size, 3, 416, 416)
		y = y.to(device) # y0 is a tensor of shape (batch_size, 9, 26, 26, 6) (num_anchors, grid_size, grid_size, 6)

		# Getting the model predictions 
		outputs = model(x) # tensor (batch_size, 9, grid_size, grid_size, num_classes + 5)
		# Calculating the loss at each scale 
		loss = loss_fn(outputs, y, scaled_anchors) # scaled_anchors is a tensor of shape (9, 2) (num_anchors, 2)

		# Add the loss to the list
		losses.append(loss.item()) 

		# Reset gradients 
		optimizer.zero_grad() 

		# Backpropagate the loss 
		loss.backward()

		# Optimization step 
		optimizer.step()

		# update progress bar with loss 
		mean_loss = sum(losses) / len(losses) 
		progress_bar.set_postfix(loss=mean_loss)


def evaluate(loader, model, loss_fn, scaled_anchors=SCALED_ANCHORS, device=DEVICE):
	"""
	Function to evaluate the model on one epoch (of the validation loader)
	"""

	model.eval()
	losses = []
	with torch.no_grad():
		for x, y in loader:
			x = x.to(device)
			y = y.to(device)
			
			outputs = model(x)
			loss = loss_fn(outputs, y, scaled_anchors)
			losses.append(loss.item())
	mean_loss = sum(losses) / len(losses) if losses else 0
	return mean_loss

# Training :

In [ ]:
# Creating the model from YOLOv3 class 
model = EfficientYOLO_12x18Grid().to(DEVICE) 
model = init_yolo_weights(model)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Nombre de paramètres entraînables : {num_params}")

# Defining the optimizer 
optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE) 

# Defining the loss function 
loss_fn = YOLOLoss() 


if LOAD_MODEL:
	load_checkpoint(CHECKPOINT_FILE, model, optimizer, LEARNING_RATE)
	
# Training the model
for e in range(1, NB_EPOCHS+1): 
	print("Epoch:", e) 
	training_loop(train_loader, model, optimizer, loss_fn, scaled_anchors=SCALED_ANCHORS, device=DEVICE) 

	# Validation
	val_loss = evaluate(val_loader, model, loss_fn, scaled_anchors=SCALED_ANCHORS, device=DEVICE)
	print(f"Validation Loss: {val_loss:.4f}")
	
	# Save validation images
	if e % 10 == 0:
		save_val_images(model, val_loader, device=DEVICE, scaled_anchors=SCALED_ANCHORS, save_dir="val_images", epoch=e, class_names = CLASS_NAMES, grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT, resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT)

	# Saving the model 
	if SAVE_MODEL: 
		save_checkpoint(model, optimizer, filename=f"checkpoint.pth.tar")

# Generating the submission CSV file
generate_submission_csv(
	model, 
	test_dataset, 
	output_csv_path="output/test_submission.csv"
)


# Check the transformations :

In [4]:
def save_gt_images(loader, class_names=CLASS_NAMES, save_dir="gt_visualizations", grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT):
	"""
	Saves images with ground truth bounding boxes from a DataLoader
	
	Args:
		loader: PyTorch DataLoader (train or val)
		class_names: List of class names
		save_dir: Directory to save visualized images
	"""
	os.makedirs(save_dir, exist_ok=True)
	print(f"Saving ground truth images to {save_dir}/")
	# Color map for different classes
	colour_map = plt.get_cmap("tab20b")
	colors = [colour_map(i) for i in np.linspace(0, 1, len(class_names))]
	
	for batch_idx, (images, targets) in enumerate(tqdm(loader)):
		# Convert images from tensor to numpy and denormalize
		images = images.permute(0, 2, 3, 1).cpu().numpy() # shape (batch_size, channels, height, width) to (batch_size, height, width, channels)
		images = (images * 255).astype(np.uint8)
		
		# Process each image in the batch
		for img_idx in range(images.shape[0]):
			img = images[img_idx]
			fig, ax = plt.subplots(1, figsize=(12, 8))
			ax.imshow(img)
			
			# Get boxes from the unique scale
			all_boxes = []
			target = targets[img_idx]
			anchor_indices = torch.where(target[..., 0] == 1)  # Cells with objects
			
			for anchor, i, j in zip(*anchor_indices):
				# for ex anchor = 0, i = 7, j = 15
				# Get box coordinates : x,y in [0,1] relative to the cell, w,h in [0,grid_width-1] and [0, grid_height-1] so relative to the grid size
				x, y, w, h = target[anchor, i, j, 1:5]
				class_id = target[anchor, i, j, 5].int() # in [0, 12]
				
				
				# Calculate box coordinates normalized to [0,1] with respect to the resized image size
				x_center = (j + x) / grid_width # normalize to [0,1]
				y_center = (i + y) / grid_height
				width = w / grid_width
				height = h / grid_height

				all_boxes.append([
					class_id.item(),  # class
					1.0,             # confidence (ground truth so 1)
					x_center.item(),
					y_center.item(),
					width.item(),
					height.item()
				])
			
			# Draw all ground truth boxes
			for box in all_boxes:
				class_id, conf, x, y, w, h = box
				
				# Calculate coordinates
				x1 = (x - w/2) * img.shape[1] # finds the upper-left corner of the box in pixels
				y1 = (y - h/2) * img.shape[0]
				width = w * img.shape[1]
				height = h * img.shape[0]
				
				# Create rectangle patch
				rect = patches.Rectangle(
					(x1, y1), width, height,
					linewidth=2,
					edgecolor=colors[int(class_id)],
					facecolor="none"
				)
				ax.add_patch(rect)
				
				# Add label
				plt.text(
					x1, y1,
					s=f"{class_names[int(class_id)]} {conf:.2f}",
					color="white",
					verticalalignment="top",
					bbox={"color": colors[int(class_id)], "pad": 0},
				)
			
			# Save image
			plt.axis("off")
			plt.savefig(
				os.path.join(save_dir, f"img_{batch_idx + img_idx}.png"),
				bbox_inches="tight",
				pad_inches=0,
				dpi=100
			)
			plt.close()

# Usage example
save_gt_images(
   loader=train_loader,
   class_names=CLASS_NAMES,
   save_dir="train_gt_visualizations"
)
print(f"Saved ground truth visualizations to train_gt_visualizations/")


NameError: name 'train_loader' is not defined

In [3]:
len(train_loader)

NameError: name 'train_loader' is not defined